# Phase 6: 백테스트 — 스윙매매 수익률 시뮬레이션

## 이 노트북에서 하는 것
각 모델의 예측값으로 실제 스윙매매를 시뮬레이션합니다.
**연간 수익률 기준으로 최고 모델을 선정합니다.**

---
### 백테스트란?
과거 데이터에서 우리 전략이 실제로 동작했다면 얼마를 벌었을지 시뮬레이션하는 것.  
모델의 **실용성**을 검증하는 핵심 단계.

### 스윙매매 전략 규칙
```
당일 시가(Open)로 매수 주문 = 예측 저가 * 0.995  (약간 여유)
당일 목표가(매도) = 예측 고가 * 1.005

체결 조건:
  - 실제 저가 ≤ 매수 주문가 → 매수 체결
  - 실제 고가 ≥ 매도 주문가 → 매도 체결
  - 당일에 둘 다 안 걸리면 → 다음날 시가에 청산 (손절)

거래비용: 편도 0.05% (Robinhood 등 무료 브로커 기준)
시드머니: $100,000
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
import os

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (14, 6)

INITIAL_CAPITAL = 100_000  # $100,000
COMMISSION      = 0.0005   # 편도 0.05%
BUY_MARGIN      = 0.995    # 예측 저가의 99.5%에 매수주문
SELL_MARGIN     = 1.005    # 예측 고가의 100.5%에 매도주문 (슬리피지 고려)

# 백테스트 결과 디렉토리
os.makedirs('/content/results', exist_ok=True)
print('백테스트 환경 세팅 완료')

In [ ]:
def run_backtest(pred_df, model_name, capital=INITIAL_CAPITAL):
    """
    스윙매매 백테스트 엔진
    
    입력: pred_df에 date, pred_high, pred_low, actual_high, actual_low, actual_open, actual_close 필요
    출력: 포트폴리오 가치 시계열 + 성능 지표
    """
    cash = capital
    shares = 0
    portfolio_values = []
    trades = []
    
    for i, row in pred_df.iterrows():
        # 현재 포트폴리오 가치
        current_price = row['actual_close']
        portfolio_val = cash + shares * current_price
        portfolio_values.append({'date': row['date'], 'value': portfolio_val})
        
        # 포지션 없을 때: 매수 시도
        if shares == 0:
            buy_price = row['pred_low'] * BUY_MARGIN
            
            # 오늘 실제 저가가 매수주문가 이하면 체결
            if row['actual_low'] <= buy_price:
                buy_price_actual = min(buy_price, row['actual_open'])  # 시가로 체결 (현실적)
                buy_price_actual = max(buy_price_actual, row['actual_low'])
                shares_bought = int(cash * 0.95 / buy_price_actual)  # 자본 95% 투입
                if shares_bought > 0:
                    cost = shares_bought * buy_price_actual * (1 + COMMISSION)
                    if cost <= cash:
                        cash -= cost
                        shares = shares_bought
                        trades.append({
                            'date': row['date'], 'type': 'BUY',
                            'price': buy_price_actual, 'shares': shares,
                            'pred_low': row['pred_low']
                        })
        
        # 포지션 있을 때: 매도 시도
        elif shares > 0:
            sell_price = row['pred_high'] * SELL_MARGIN
            
            # 오늘 실제 고가가 매도주문가 이상이면 체결
            if row['actual_high'] >= sell_price:
                sell_price_actual = min(sell_price, row['actual_high'])
                revenue = shares * sell_price_actual * (1 - COMMISSION)
                cash += revenue
                trades.append({
                    'date': row['date'], 'type': 'SELL',
                    'price': sell_price_actual, 'shares': shares,
                    'pred_high': row['pred_high']
                })
                shares = 0
            else:
                # 다음날 시가에 강제 청산 (최대 1일 보유)
                if i < len(pred_df) - 1:
                    next_open = pred_df.iloc[pred_df.index.get_loc(i)+1]['actual_open'] if isinstance(i, int) else current_price
                    revenue = shares * current_price * (1 - COMMISSION)
                    cash += revenue
                    trades.append({
                        'date': row['date'], 'type': 'FORCE_SELL',
                        'price': current_price, 'shares': shares
                    })
                    shares = 0
    
    portfolio_df = pd.DataFrame(portfolio_values)
    portfolio_df['date'] = pd.to_datetime(portfolio_df['date'])
    portfolio_df = portfolio_df.set_index('date')
    
    trades_df = pd.DataFrame(trades) if trades else pd.DataFrame()
    
    # 성능 지표 계산
    final_value = cash + shares * pred_df.iloc[-1]['actual_close']
    total_return = (final_value - capital) / capital * 100
    
    # 샤프 지수 (위험 대비 수익)
    daily_returns = portfolio_df['value'].pct_change().dropna()
    sharpe = (daily_returns.mean() / daily_returns.std()) * np.sqrt(252) if daily_returns.std() > 0 else 0
    
    # 최대 낙폭 (MDD)
    rolling_max = portfolio_df['value'].cummax()
    drawdown = (portfolio_df['value'] - rolling_max) / rolling_max * 100
    mdd = drawdown.min()
    
    # 거래 통계
    buy_trades  = [t for t in trades if t['type'] == 'BUY']
    sell_trades = [t for t in trades if t['type'] == 'SELL']
    
    metrics = {
        'model': model_name,
        'initial': capital,
        'final': round(final_value, 2),
        'total_return(%)': round(total_return, 2),
        'sharpe_ratio': round(sharpe, 3),
        'MDD(%)': round(mdd, 2),
        'total_trades': len(buy_trades),
        'win_rate(%)': round(len(sell_trades) / max(len(buy_trades), 1) * 100, 1)
    }
    
    print(f'\n[{model_name}] 최종: ${final_value:,.0f} | 수익률: {total_return:.1f}% | Sharpe: {sharpe:.2f} | MDD: {mdd:.1f}% | 거래: {len(buy_trades)}회')
    
    return portfolio_df, trades_df, metrics

print('백테스트 엔진 준비 완료')

In [ ]:
# 각 모델 예측 파일 로드 + 백테스트 실행
model_files = {
    'ARIMA':   '/content/results/arima_predictions.csv',
    'Prophet': '/content/results/prophet_predictions.csv',
    'XGBoost': '/content/results/xgboost_predictions.csv',
    'LSTM':    '/content/results/lstm_predictions.csv',
}

bt_results = {}
all_metrics = []

for model_name, filepath in model_files.items():
    if not os.path.exists(filepath):
        print(f'{model_name}: 파일 없음 ({filepath}), 스킵')
        continue
    
    pred_df = pd.read_csv(filepath)
    # actual_open이 없으면 actual_close 사용
    if 'actual_open' not in pred_df.columns:
        pred_df['actual_open'] = pred_df['actual_close']
    
    pred_df = pred_df.reset_index(drop=True)
    
    portfolio_df, trades_df, metrics = run_backtest(pred_df, model_name)
    bt_results[model_name] = (portfolio_df, trades_df, metrics)
    all_metrics.append(metrics)

In [ ]:
# 포트폴리오 성장 비교 차트 (발표 핵심 슬라이드)
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

colors = {'ARIMA': '#ffd700', 'Prophet': '#26a69a', 'XGBoost': 'dodgerblue', 'LSTM': '#ef5350', 'TFT': '#ff6b35'}

for model_name, (portfolio_df, _, metrics) in bt_results.items():
    axes[0].plot(portfolio_df.index, portfolio_df['value'],
                 label=f"{model_name} ({metrics['total_return(%)']:+.1f}%)",
                 color=colors.get(model_name, 'white'), lw=2)

axes[0].axhline(y=INITIAL_CAPITAL, color='gray', ls='--', lw=1, label='시드머니 $100,000')
axes[0].set_title('모델별 포트폴리오 성장 비교', fontsize=14)
axes[0].set_ylabel('포트폴리오 가치 (USD)')
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 수익률 바 차트
if all_metrics:
    metrics_df = pd.DataFrame(all_metrics).sort_values('total_return(%)', ascending=True)
    bar_colors = ['#ef5350' if v < 0 else '#26a69a' for v in metrics_df['total_return(%)']]
    bars = axes[1].barh(metrics_df['model'], metrics_df['total_return(%)'], color=bar_colors)
    axes[1].axvline(x=0, color='white', lw=0.5)
    for bar, val in zip(bars, metrics_df['total_return(%)']):
        axes[1].text(val + (0.5 if val >= 0 else -0.5), bar.get_y() + bar.get_height()/2,
                     f'{val:+.1f}%', va='center', ha='left' if val >= 0 else 'right', fontweight='bold')
    axes[1].set_title('연간 수익률 비교 (백테스트)', fontsize=14)
    axes[1].set_xlabel('수익률 (%)')
    axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('/content/backtest_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 종합 성능 테이블 출력
if all_metrics:
    metrics_df = pd.DataFrame(all_metrics)
    metrics_df = metrics_df.sort_values('total_return(%)', ascending=False)
    metrics_df['final'] = metrics_df['final'].apply(lambda x: f'${x:,.0f}')
    
    print('\n' + '='*80)
    print('최종 백테스트 성능 비교표 ($100,000 시드 기준)')
    print('='*80)
    print(metrics_df.to_string(index=False))
    print('='*80)
    
    best_model = metrics_df.iloc[0]['model']
    best_return = metrics_df.iloc[0]['total_return(%)']
    print(f'\n최고 수익률 모델: {best_model} ({best_return:+.1f}%)')
    
    metrics_df.to_csv('/content/results/backtest_summary.csv', index=False)
    print('결과 저장 완료!')

In [ ]:
# 최고 모델의 거래 내역 시각화
if bt_results:
    best_model_name = pd.DataFrame(all_metrics).sort_values('total_return(%)', ascending=False).iloc[0]['model']
    best_portfolio, best_trades, best_metrics = bt_results[best_model_name]
    
    print(f'\n최고 모델 [{best_model_name}] 거래 내역 분석')
    print(f'총 거래: {best_metrics["total_trades"]}회')
    
    if len(best_trades) > 0:
        buy_trades  = best_trades[best_trades['type'] == 'BUY']
        sell_trades = best_trades[best_trades['type'].isin(['SELL', 'FORCE_SELL'])]
        
        # MDD 계산
        rolling_max = best_portfolio['value'].cummax()
        drawdown = (best_portfolio['value'] - rolling_max) / rolling_max * 100
        
        fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
        
        axes[0].plot(best_portfolio.index, best_portfolio['value'], color='#ffd700', lw=1.5, label='포트폴리오 가치')
        axes[0].set_ylabel('가치 (USD)')
        axes[0].set_title(f'[{best_model_name}] 최고 수익률 모델 상세 분석')
        axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
        axes[0].grid(True, alpha=0.3)
        axes[0].legend()
        
        axes[1].fill_between(drawdown.index, drawdown.values, 0, alpha=0.5, color='#ef5350', label='낙폭')
        axes[1].set_ylabel('낙폭 (%)')
        axes[1].set_title('최대 낙폭 (MDD)')
        axes[1].grid(True, alpha=0.3)
        axes[1].legend()
        
        plt.tight_layout()
        plt.savefig('/content/best_model_detail.png', dpi=150, bbox_inches='tight')
        plt.show()

print('\n다음: 07_final_prediction.ipynb — 실제 미래 예측!')